# Malaika Edge — Fully Offline IMCI Assessment

**ZERO internet. ZERO cloud calls. Everything runs on this machine.**

This notebook proves Malaika works completely offline:
- **STT**: Whisper-small (244 MB, local)
- **Inference**: Gemma 4 E4B via Unsloth (local GPU)
- **TTS**: Piper TTS (offline, 4 languages)
- **UI**: Same voice UI served locally via FastAPI

No Smallest AI. No ngrok. No API keys needed (only HF_TOKEN for initial model download).

**After the model downloads once, disconnect WiFi and it still works.**

In [ ]:
# Cell 1: Install deps (kernel restarts at end)
# These are the ONLY downloads that need internet.
# After this cell, everything runs offline.

!rm -rf /content/malaika
!git clone https://github.com/klickgenai/deepmind-hackathon.git /content/malaika
%cd /content/malaika
!git checkout edge-offline

!pip install -q unsloth
!pip install -q --no-deps trl datasets librosa soundfile Pillow
!pip install -q structlog pyyaml httpx websockets fastapi uvicorn
!pip install -q piper-tts  # Offline TTS

# Restart kernel
import os; os._exit(0)

In [ ]:
# Cell 2: Load model + Whisper (last step needing internet for model download)
# After this cell completes, you can disconnect WiFi.

%cd /content/malaika
import sys, os, time
sys.path.insert(0, '/content/malaika')

# --- HuggingFace auth (for model download only) ---
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print("HF auth: OK")
except Exception:
    login()

# --- Load Gemma 4 E4B ---
import torch
from unsloth import FastModel

print("\nLoading Gemma 4 E4B (this is the last download)...")
t0 = time.time()
model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-E4B-it",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
    full_finetuning=False,
)
print(f"Gemma 4 loaded in {time.time()-t0:.0f}s | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained("google/gemma-4-E4B-it")
print("Processor loaded")

# --- Load Whisper for offline STT ---
print("\nLoading Whisper-small for offline STT...")
t0 = time.time()
from malaika.audio import WhisperTranscriber
whisper = WhisperTranscriber()
whisper._load()  # Force download now while we have internet
print(f"Whisper loaded in {time.time()-t0:.0f}s")

# --- Quick vision test ---
from PIL import Image
img = Image.new("RGB", (128, 128), color=(200, 150, 100))
messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "What color? One word."}]}]
input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=input_text, images=[img], return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=10, do_sample=False, repetition_penalty=1.3)
resp = processor.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(f"Vision test: '{resp}'")

print("\n" + "="*50)
print("  ALL MODELS LOADED. YOU CAN NOW GO OFFLINE.")
print("  Disconnect WiFi to prove it works without internet.")
print("="*50)

In [ ]:
# Cell 3: Launch edge app (FULLY OFFLINE from here)
# No ngrok, no cloud APIs, no internet needed.

from malaika.edge_app import create_edge_app
from malaika.skills import SkillRegistry

# Create fully offline app
edge_app = create_edge_app(model=model, processor=processor)
print(f"Skills: {len(SkillRegistry.list_all())}")

# --- Serve locally ---
# On Colab, we use the built-in port forwarding (no ngrok needed)
import threading
import uvicorn

PORT = 8000

def run_server():
    uvicorn.run(edge_app, host="0.0.0.0", port=PORT, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

import time; time.sleep(2)

# Colab provides automatic port forwarding for localhost
from google.colab.output import eval_js
colab_url = eval_js(f'google.colab.kernel.proxyPort({PORT})')

print(f"\n{'='*60}")
print(f"  MALAIKA EDGE IS LIVE — FULLY OFFLINE")
print(f"{'='*60}")
print(f"\n  Open in browser:")
print(f"  {colab_url}")
print(f"\n  Everything runs on this machine:")
print(f"  - STT:       Whisper-small (local)")
print(f"  - Inference:  Gemma 4 E4B (local GPU)")
print(f"  - TTS:       Piper (local)")
print(f"  - Internet:  NOT NEEDED")
print(f"\n  Features:")
print(f"  - Voice:  Tap the orb to talk (Whisper transcribes)")
print(f"  - Text:   Type messages")
print(f"  - Camera: Take photos for vision analysis")
print(f"  - IMCI:   Skill cards + WHO classifications")
print(f"\n{'='*60}")
print(f"  Disconnect WiFi now to prove offline capability.")
print(f"{'='*60}")

# Keep alive
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\nShutdown.")